In [1]:
# Comandos para baixar os "dicionários" que o Python precisa
import nltk
import string # Importar o módulo string
from nltk.corpus import stopwords # Importar stopwords

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab') # Esta é a linha que estava faltando!

# Baixar recursos necessários para processar texto em português
nltk.download('stopwords')
nltk.download('punkt')

# Criar uma lista de palavras para ignorar ignorar (artigos, preposições)
ponto = list(string.punctuation)
palavras_irrelevantes = stopwords.words('portuguese')
todas_stop_words = ponto + palavras_irrelevantes

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
import pandas as pd # Importar a biblioteca pandas

# Criando dados de exemplo (simulando o que vem do Excel/Kaggle)
dados = {
    'texto': [
        'O produto chegou muito rápido e é ótimo!',
        'Péssimo, veio quebrado e demorou um mês.',
        'Gostei bastante, recomendo a todos.',
        'Não recebi o produto, estou muito insatisfeito.',
        'A qualidade é boa, mas a cor é diferente da foto.'
    ],
    'nota': [5, 1, 5, 1, 3]
}

df = pd.DataFrame(dados)
print("Dados carregados com sucesso!")
print(df)

Dados carregados com sucesso!
                                               texto  nota
0           O produto chegou muito rápido e é ótimo!     5
1           Péssimo, veio quebrado e demorou um mês.     1
2                Gostei bastante, recomendo a todos.     5
3    Não recebi o produto, estou muito insatisfeito.     1
4  A qualidade é boa, mas a cor é diferente da foto.     3


In [3]:
def limpar_texto(frase):
    # 1. Deixar tudo em minúsculo
    frase = frase.lower()

    # 2. Quebrar a frase em palavras individuais (tokenização)
    palavras = nltk.word_tokenize(frase)

    # 3. Filtrar: só manter palavras que NÃO estão na nossa lista de "lixo"
    limpas = [p for p in palavras if p not in todas_stop_words]

    # 4. Juntar as palavras de volta em uma frase limpa
    return " ".join(limpas)

# Aplicar a limpeza na  tabela
df['texto_limpo'] = df['texto'].apply(limpar_texto)

print("Texto antes vs Texto depois:")
print(df[['texto', 'texto_limpo']])

Texto antes vs Texto depois:
                                               texto  \
0           O produto chegou muito rápido e é ótimo!   
1           Péssimo, veio quebrado e demorou um mês.   
2                Gostei bastante, recomendo a todos.   
3    Não recebi o produto, estou muito insatisfeito.   
4  A qualidade é boa, mas a cor é diferente da foto.   

                         texto_limpo  
0        produto chegou rápido ótimo  
1  péssimo veio quebrado demorou mês  
2    gostei bastante recomendo todos  
3        recebi produto insatisfeito  
4   qualidade boa cor diferente foto  


In [4]:
# Baixar recursos necessários para processar texto em português
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [5]:
# 1. Primeiro, descobrir qual nome  para a  tabela
# (Isso evita o erro de 'name not defined')
if 'df_limpo' in locals() and df_limpo is not None:
    dados_trabalho = df_limpo.copy() # Use copy to avoid modifying original
    # If df_limpo is already fully prepared with 'sentimento' and 'texto_tratado', no further action needed here.
    # Otherwise, add logic to create them if missing.
    if 'texto_tratado' not in dados_trabalho.columns and 'texto_limpo' in dados_trabalho.columns:
        dados_trabalho['texto_tratado'] = dados_trabalho['texto_limpo']
    if 'sentimento' not in dados_trabalho.columns:
        if 'nota' in dados_trabalho.columns:
            dados_trabalho['sentimento'] = dados_trabalho['nota'].apply(lambda x: 1 if x >= 4 else 0)
        elif 'review_score' in dados_trabalho.columns:
            dados_trabalho['sentimento'] = dados_trabalho['review_score'].apply(lambda x: 1 if x >= 4 else 0)

elif 'df' in locals() and df is not None:
    dados_trabalho = df.copy() # Added .copy() to prevent SettingWithCopyWarning
    # Ensure 'texto_tratado' column exists using 'texto_limpo' from df
    dados_trabalho['texto_tratado'] = dados_trabalho['texto_limpo']
    # Create 'sentimento' based on 'nota' column (since df has 'nota', not 'review_score')
    dados_trabalho['sentimento'] = dados_trabalho['nota'].apply(lambda x: 1 if x >= 4 else 0)

else:
    print("❌ Erro: Você ainda não carregou os dados do arquivo CSV!")
    # Caso não exista nenhum dos dois,  tentar carregar agora:
    df_real = pd.read_csv('olist_order_reviews_dataset.csv')
    dados_trabalho = df_real[['review_comment_message', 'review_score']].dropna(subset=['review_comment_message']).copy() # Added .copy()
    # Aplicar a limpeza que definimos antes
    dados_trabalho['texto_tratado'] = dados_trabalho['review_comment_message'].apply(limpar_texto)
    # Create 'sentimento' based on 'review_score' column
    dados_trabalho['sentimento'] = dados_trabalho['review_score'].apply(lambda x: 1 if x >= 4 else 0)

# 3. Importar as ferramentas de Inteligência Artificial
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# 4. Criar o "Tradutor" (Vetorizador)
vetorizador = TfidfVectorizer(max_features=500)

# 5. Transformar texto em números
X = vetorizador.fit_transform(dados_trabalho['texto_tratado'])
y = dados_trabalho['sentimento']

# 6. Separar Treino e Teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ Sucesso! Dados preparados e nomes corrigidos.")
print(f"Temos {X_treino.shape[0]} exemplos para o robô estudar.")

✅ Sucesso! Dados preparados e nomes corrigidos.
Temos 4 exemplos para o robô estudar.


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Criar o modelo (o robô vazio)
modelo = LogisticRegression()


# O robô vai olhar para X_treino (palavras) e y_treino (sentimento)
modelo.fit(X_treino, y_treino)


# Pedir para ele prever os resultados dos dados que guardamos para teste
previsoes = modelo.predict(X_teste)

# 4. Calcular a nota do robô
acuracia = accuracy_score(y_teste, previsoes)

print(f"O robô terminou o treino! A precisão dele foi de: {acuracia * 100:.2f}%")

O robô terminou o treino! A precisão dele foi de: 100.00%


In [7]:
# 1.frase entre as aspas:
frase_do_usuario = "Gostei bastante, recomendo a todos"

# 2.  limpar a sua frase igual limpamos os dados do Kaggle
frase_limpa = limpar_texto(frase_do_usuario)

# 3. Transformar a frase limpa em números (vetorização)
frase_vetorizada = vetorizador.transform([frase_limpa])

# 4. Pedir para o robô dar o veredito
predicao = modelo.predict(frase_vetorizada)

# 5. Mostrar o resultado de um jeito bonito
if predicao[0] == 1:
    print(f"Mensagem: {frase_do_usuario}")
    print("Análise do Robô: POSITIVO 😊 (O cliente parece satisfeito)")
else:
    print(f"Mensagem: {frase_do_usuario}")
    print("Análise do Robô: NEGATIVO 😡 (O cliente parece insatisfeito)")

Mensagem: Gostei bastante, recomendo a todos
Análise do Robô: POSITIVO 😊 (O cliente parece satisfeito)
